# 08 — Formulación MDP de las otras tareas del catálogo

El catálogo de P1 incluye cuatro tareas acotadas. La persecución de balón está implementada y evaluada
(notebooks 01–07). Aquí dejamos **formulado** el MDP de las otras tres, con los **mismos supuestos simples del
starter kit**:
- paso de tiempo discreto; `DASH` mueve al jugador una distancia exacta y `TURN` gira exactamente ±35°;
- `KICK` desplaza el balón una distancia fija en la dirección del jugador, solo si está en posesión
  ($d_b \le 0.8$ m);
- sin inercia ni ruido de movimiento, con observación completa y en la cancha de 105 × 68 m (salvo el 2v1, que
  usa una zona de 30 × 20 m).

**Alcance:** es una formulación propuesta. En esta entrega no está implementada ni entrenada.

Criterio común de discretización, igual que en la persecución:
- 17.5° es la mitad del giro de 35°, el error máximo tras girar;
- las zonas de distancia reflejan pasos restantes;
- 0.8 m es el radio de posesión.

## 1. Conducción y drible

**Reto.** El agente empieza con el balón al pie y debe avanzar hacia el arco rival con micro-pateos y carreras
cortas, sin perder el balón ni salir de la cancha. Inicio: jugador y balón en $x_0 \sim U[-40, -10]$ m e
$y_0 \sim U[-20, 20]$ m, con orientación aleatoria.

| elemento | definición |
|---|---|
| $\mathcal{S}$ | $d_b$ (distancia al balón): ≤ 0.8 (posesión), (0.8, 2], (2, 4] m · $\theta_b$ (rumbo al balón): frente ±17.5°, izquierda, derecha · $\theta_g$ (rumbo al arco): frente ±17.5°, izquierda, derecha, atrás (> 90°) · $d_g$ (distancia al arco): < 20, 20–40, ≥ 40 m |
| $\mathcal{A}$ | `KICK 25`: si $d_b \le 0.8$, el balón avanza 2 m en la dirección del jugador · `DASH 80`: el jugador avanza 0.8 m · `TURN +35` / `TURN −35` |
| $\mathcal{P}$ | Determinista, con la cinemática del kit. Si $d_b > 0.8$, `KICK` no tiene efecto |
| $\mathcal{R}$ | $\Delta x$ del balón hacia el arco − 0.1 por paso; −30 si $d_b > 4$ m (pérdida) o si el balón o el jugador salen de la cancha; +50 al acumular 30 m de avance |
| $\gamma$, $T_{max}$ | 0.99; 100 pasos |
| terminales | pérdida, salida, 30 m de avance |
| criterio | avance > 30 m manteniendo el control en > 80 % de los episodios |

**Justificación.** 0.8 m separa tener el balón al pie, donde se puede patear, de tenerlo cerca, donde hay que
alcanzarlo. 4 m marca la pérdida de control. Avanzar 2 m cuesta un `KICK` y unos 2–3 `DASH`, así que 30 m
requieren ≈ 50 pasos, dentro de $T_{max}$.

## 2. Tiro a puerta

**Reto.** El atacante, con el balón frente al arco rival, con o sin arquero, elige hacia dónde tirar. Inicio:
distancia al centro del arco $d_g \sim U[11, 25]$ m y desplazamiento lateral $y \sim U[-10, 10]$ m. El arco mide
14.02 m (postes en $y = \pm 7.01$ m). Si hay arquero, se ubica en la línea de gol en $y_k \sim U[-3, 3]$ m.

| elemento | definición |
|---|---|
| $\mathcal{S}$ | $d_g$: 11–15, 15–20, 20–25 m · posición lateral: izquierda, centro (\|y\| ≤ 3.5 m), derecha · arquero: sin arquero, a la izquierda, al centro, a la derecha |
| $\mathcal{A}$ | tiro a una de 5 zonas del arco (poste izquierdo, izquierda, centro, derecha, poste derecho) · `CONDUCIR` (el jugador y el balón avanzan 2 m hacia el arco) |
| $\mathcal{P}$ | El tiro llega a la línea de gol con un error lateral $\epsilon \sim \mathcal{N}(0, (0.05\,d_g)^2)$ m. Es gol si pasa entre los postes y el arquero no lo ataja (ataja si \|y − y_k\| < 1 m) |
| $\mathcal{R}$ | +100 gol, −30 tiro afuera, −50 tiro atajado (valores del enunciado); −0.2 por `CONDUCIR` |
| $\gamma$, $T_{max}$ | 0.99; 10 pasos (el episodio termina al tirar) |
| criterio | conversión > 75 % sin arquero y > 50 % con arquero |

**Justificación.** La distancia y la posición lateral determinan los ángulos a los dos postes
$(\theta_{post\_l}, \theta_{post\_r})$, así que representarlas es equivalente con menos estados. El error del
tiro crece con la distancia, y eso hace que `CONDUCIR` para acercarse tenga sentido.

## 3. Cooperación 2v1 y pase

**Reto.** Dos atacantes deben retener el balón frente a un defensor pasivo o estocástico, eligiendo entre
conducir y pasar al compañero libre. Zona de 30 × 20 m. El agente controla siempre al atacante con balón; el
compañero se ubica en un espacio libre (regla fija) y el defensor presiona al poseedor.

| elemento | definición |
|---|---|
| $\mathcal{S}$ | $d_{comp}$: < 5, 5–10, ≥ 10 m · $\theta_{comp}$: frente, lado, atrás · $d_{def}$: < 2, 2–5, ≥ 5 m · $\theta_{def}$: frente, lado, atrás · línea de pase bloqueada (defensor a < 1.5 m del segmento poseedor–compañero): sí/no · zona del balón: centro / cerca de la banda |
| $\mathcal{A}$ | `PASE` al compañero · `DRIBLE` (avanza 1 m con el balón) · `GIRAR` (35° alejándose del defensor) · `DESPEJE` (patea lejos y termina la posesión) |
| $\mathcal{P}$ | El defensor avanza 0.6 m/paso hacia el poseedor. Con la línea bloqueada, el pase se intercepta con prob. 0.8. Si $d_{def} < 1$ m, quite con prob. 0.5. Tras un pase completado, el agente pasa a controlar al nuevo poseedor |
| $\mathcal{R}$ | +30 pase completado, −30 intercepción o quite, −10 balón fuera de la zona (valores del enunciado); `DESPEJE` termina con 0 |
| $\gamma$, $T_{max}$ | 0.99; 60 pasos |
| terminales | intercepción, salida, despeje |
| criterio | posesión > 50 pasos y al menos 3 pases efectivos |

**Justificación.** El riesgo de intercepción lo decide casi por completo la posición del defensor respecto de la
línea de pase. Un bit que lo indica reemplaza la geometría completa, y eso mantiene la tabla chica.

## 4. Dimensionamiento de las tablas Q

In [1]:
from math import prod
from IPython.display import display, Markdown

tasks = {
    "Persecución (implementada, R3)": ({"d": 9, "θ": 11}, 4),
    "Conducción y drible": ({"d_b": 3, "θ_b": 3, "θ_g": 4, "d_g": 3}, 4),
    "Tiro a puerta": ({"d_g": 3, "lateral": 3, "arquero": 4}, 6),
    "Cooperación 2v1": ({"d_comp": 3, "θ_comp": 3, "d_def": 3, "θ_def": 3, "línea bloqueada": 2, "zona": 2}, 4),
}
rows = ["| tarea | variables (bins) | \\|S\\| | \\|A\\| | valores Q |", "|---|---|---|---|---|"]
for name, (vars_, n_actions) in tasks.items():
    n_states = prod(vars_.values())
    desc = ", ".join(f"{k} ({v})" for k, v in vars_.items())
    rows.append(f"| {name} | {desc} | {n_states} | {n_actions} | {n_states * n_actions} |")
display(Markdown("\n".join(rows)))

| tarea | variables (bins) | \|S\| | \|A\| | valores Q |
|---|---|---|---|---|
| Persecución (implementada, R3) | d (9), θ (11) | 99 | 4 | 396 |
| Conducción y drible | d_b (3), θ_b (3), θ_g (4), d_g (3) | 108 | 4 | 432 |
| Tiro a puerta | d_g (3), lateral (3), arquero (4) | 36 | 6 | 216 |
| Cooperación 2v1 | d_comp (3), θ_comp (3), d_def (3), θ_def (3), línea bloqueada (2), zona (2) | 324 | 4 | 1296 |

Las cuatro tablas son manejables con los algoritmos tabulares de P1 (MC, SARSA, Q-Learning). La más grande, la
del 2v1, tiene 1 296 valores. Implementarlas requiere un entorno simple por tarea, con la misma interfaz
`reset`/`step` que `src/kit_env.py`; el resto del pipeline (agentes, entrenamiento, evaluación) se reutiliza.